# CENG 476 — Plant Disease Classification Using Deep Learning and Transfer Learning

**Student:** Emir Evren — **ID:** 210444038  
**Framework:** PyTorch  
**Task:** 38-class PlantVillage image classification

This notebook is organized as a **model-development and training-methodology notebook**. The focus is not architectural novelty; it is the complete training process: preprocessing, data augmentation, Batch Normalization, regularization, loss, optimizer, learning-rate strategy, scheduler, early stopping, train/validation behavior, transfer learning, evaluation metrics and the effect of the major experimental decisions.

The expensive GPU runs are not automatically repeated when the notebook is opened. Code cells show the implementation and recorded outputs from completed runs are retained so that the methodology and results can be inspected. Repository scripts and CSV/JSON outputs remain the source of truth.

**Final audited benchmark:** Custom CNN **84.62%**, ResNet18 **97.66%**, EfficientNet-B0 **99.01%**, validation-selected 50/50 ensemble **99.14%**.

## 1. Requirement Coverage

| Project requirement | Implementation in this project |
|---|---|
| Custom architecture | Four-block CNN trained from scratch |
| Architecture explanation | Conv-BN-ReLU-MaxPool blocks, 3×3 kernels, adaptive pooling |
| Batch Normalization | after every convolution in the Custom CNN |
| Dropout | 0.40 baseline; 0.30 transfer classifiers |
| Regularization | data augmentation + dropout + AdamW weight decay `1e-4` |
| Over/underfitting analysis | augmented train + clean train + validation + test |
| Optimizer | AdamW, betas `(0.9,0.999)` |
| LR tuning | baseline pilots `1e-3`, `5e-4`, `3e-4` |
| Transfer LR strategy | backbone `1e-4`, classifier `5e-4` |
| Scheduler | ReduceLROnPlateau |
| Early stopping | patience 6 baseline, 5 transfer |
| Activations | ReLU; EfficientNet native SiLU |
| Evaluation | fixed train / validation / locked test; no k-fold CV |
| Metrics | accuracy, precision, recall, sensitivity, specificity, Macro/Weighted F1, ROC-AUC, confusion matrix |
| Creative experiment | validation-selected soft-voting ensemble |
| Reproducibility | seed 42 + seeds 123/777 stability check |

The sections below follow this checklist and explicitly state **what was used, why it was used, and what was observed**.

## 2. Project Paths and Reproducibility

The reference seed is **42**. Python, NumPy and PyTorch are seeded. On CUDA, all GPU seeds are set and deterministic cuDNN behavior is requested. The seed is not special; it gives a reproducible reference run.

In [1]:
from pathlib import Path
import random, sys
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print('Reference seed:', SEED)
print('Python / NumPy / PyTorch seeded: yes')

Reference seed: 42
Python / NumPy / PyTorch seeded: yes


## 3. Dataset and Three-Way Split

The problem is **single-label, 38-class image classification**. A fixed **train / validation / test** protocol is used.

- **Train:** supplies gradients and receives random augmentation.
- **Validation:** used for scheduler decisions, checkpoint selection, hyperparameter analysis and ensemble-weight selection.
- **Locked test:** used for final evaluation.

**K-fold cross-validation was not used.** Repeating full CNN fine-tuning across multiple folds would substantially increase compute cost. A dedicated validation split and locked test were already maintained, and reproducibility was separately evaluated with multiple random seeds.

The locked test set was not used for model training, checkpoint selection, hyperparameter tuning or ensemble-weight selection. Test images participated only in deterministic, model-independent duplicate / near-duplicate integrity auditing.

In [2]:
initial = {'train': 43444, 'validation': 5430, 'test': 5431}
final = {'train': 39091, 'validation': 4462, 'test': 10709}

print('INITIAL IMAGE-LEVEL PROTOCOL')
for k, v in initial.items():
    print(f'{k:10s}: {v:,}')
print('total     :', f'{sum(initial.values()):,}')

print('\nFINAL ULTRA-STRICT PROTOCOL')
for k, v in final.items():
    print(f'{k:10s}: {v:,}')
print('total     :', f'{sum(final.values()):,}')
print('classes   : 38')

INITIAL IMAGE-LEVEL PROTOCOL
train     : 43,444
validation: 5,430
test      : 5,431
total     : 54,305

FINAL ULTRA-STRICT PROTOCOL
train     : 39,091
validation: 4,462
test      : 10,709
total     : 54,262
classes   : 38


## 4. Preprocessing and Data Augmentation

### Why augmentation is used
Training-only augmentation introduces controlled variation so the model is less dependent on one exact crop, orientation or illumination condition. The selected transformations are moderate because plant-disease classification depends strongly on lesion color and fine texture.

### Training transformations
- `RandomResizedCrop(224, scale=(0.80,1.00))`: changes crop/scale while keeping a 224×224 input.
- `RandomHorizontalFlip(p=0.5)`: removes unnecessary left/right orientation dependence.
- `RandomRotation(15)`: introduces rotations from approximately -15° to +15°.
- `ColorJitter(0.2)`: moderate brightness, contrast and saturation variation.
- ImageNet normalization: required for compatibility with ImageNet-pretrained transfer models.

### Validation/test preprocessing
`Resize(256) → CenterCrop(224) → ToTensor() → Normalize`. These transforms are deterministic; random evaluation transforms would add measurement noise.

**Claim boundary:** an augmentation-on/off single-variable ablation was not performed, so no isolated percentage-point gain is attributed to augmentation alone.

In [3]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.80, 1.00)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(
        15, interpolation=InterpolationMode.BILINEAR, fill=(128, 128, 128)
    ),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

evaluation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('TRAIN: RandomResizedCrop(224, 0.80-1.00) + HFlip(0.5) + Rotation(±15°) + ColorJitter(0.2) + ImageNet Normalize')
print('EVAL : Resize(256) + CenterCrop(224) + ImageNet Normalize')

TRAIN: RandomResizedCrop(224, 0.80-1.00) + HFlip(0.5) + Rotation(±15°) + ColorJitter(0.2) + ImageNet Normalize
EVAL : Resize(256) + CenterCrop(224) + ImageNet Normalize


## 5. Input Dimensions and DataLoader Choices

All models receive RGB images of shape `3×224×224`. The Custom CNN uses batch size **64**; transfer models use **32**. The smaller transfer-model batch helps control GPU memory use.

A deterministic **clean-train evaluation subset** contains 20 images per class, therefore `38 × 20 = 760` images. It is not used for gradient updates; it is used to compare train/validation behavior without random augmentation.

In [4]:
print('Baseline batch : [64, 3, 224, 224]')
print('Transfer batch : [32, 3, 224, 224]')
print('Output logits  : [B, 38]')
print('Clean-train eval images:', 38 * 20)

Baseline batch : [64, 3, 224, 224]
Transfer batch : [32, 3, 224, 224]
Output logits  : [B, 38]
Clean-train eval images: 760


## 6. Custom CNN Architecture

The scratch baseline contains four repeated `Conv → BatchNorm → ReLU → MaxPool` blocks. Channel depth increases `32 → 64 → 128 → 256`. The feature extractor is followed by `AdaptiveAvgPool(1×1)`, `Dropout(0.40)` and `Linear(256,38)`.

**Why this architecture?** Convolution is suited to spatially local structures such as lesions, texture and discoloration. A 3×3 kernel is a compact standard choice for local visual features. `padding=1` preserves spatial size through each convolution; max pooling performs controlled downsampling. Adaptive pooling removes dependence on the exact final feature-map dimensions and keeps the classifier small.

In [5]:
from torch import nn

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=38, dropout_rate=0.40):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

baseline = BaselineCNN()
x = torch.randn(2, 3, 224, 224)
with torch.inference_mode():
    y = baseline.eval()(x)
params = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
print('Input shape :', tuple(x.shape))
print('Output shape:', tuple(y.shape))
print('Trainable parameters:', f'{params:,}')

Input shape : (2, 3, 224, 224)
Output shape: (2, 38)
Trainable parameters: 399,142


## 7. Batch Normalization, Dropout and Regularization

**Batch Normalization** is placed after each convolution and before ReLU. During training it uses mini-batch statistics; during evaluation `model.eval()` makes it use stored running statistics. Its purpose here is optimization stability, not to create a new architecture.

**Dropout** randomly disables activations only during training. The scratch classifier uses **0.40** and transfer classifiers use **0.30**. `model.eval()` disables dropout during validation/test.

Other explicit regularization methods are training-only augmentation and AdamW weight decay `1e-4`.

In [6]:
bn_layers = [m for m in baseline.modules() if isinstance(m, nn.BatchNorm2d)]
print('BatchNorm layers in Custom CNN:', len(bn_layers))
for dropout, score in [(0.20, 0.4990), (0.40, 0.4953), (0.60, 0.4311)]:
    print(f'dropout={dropout:.2f} -> best validation Macro-F1={score:.4f}')

BatchNorm layers in Custom CNN: 4
dropout=0.20 -> best validation Macro-F1=0.4990
dropout=0.40 -> best validation Macro-F1=0.4953
dropout=0.60 -> best validation Macro-F1=0.4311


The controlled dropout pilot shows that **0.60 was too aggressive** and suppressed learning. The difference between 0.20 and 0.40 was small in the short pilot. The predefined final baseline remained at 0.40; test performance was not used to retroactively select dropout.

## 8. Activation Functions, Logits, Softmax and Loss

The Custom CNN and ResNet18 use **ReLU**; EfficientNet-B0 retains its native **SiLU**. Nonlinearity is required because stacking only linear transformations is still equivalent to a single linear transformation.

The output layer returns **38 raw logits**. Softmax is intentionally not inserted before the loss. `nn.CrossEntropyLoss()` expects raw logits and internally performs the stable LogSoftmax / negative-log-likelihood computation. Softmax is used later only when probabilities are required.

In [7]:
criterion = nn.CrossEntropyLoss()
example_logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.8, 0.2]])
example_labels = torch.tensor([0, 1])
example_loss = criterion(example_logits, example_labels)
example_probs = torch.softmax(example_logits, dim=1)
print('CrossEntropyLoss:', f'{example_loss.item():.4f}')
print('Softmax probabilities for sample 1:', [round(v, 4) for v in example_probs[0].tolist()])
print('Probability sum:', f'{example_probs[0].sum().item():.4f}')

CrossEntropyLoss: 0.3161
Softmax probabilities for sample 1: [0.7856, 0.1753, 0.0391]
Probability sum: 1.0000


## 9. AdamW, Learning Rates, Scheduler and Early Stopping

All final models use **AdamW**, `betas=(0.9,0.999)` and `weight_decay=1e-4`. AdamW provides adaptive updates while decoupling weight decay from the adaptive gradient step.

Baseline LR pilots: `1e-3`, `5e-4`, `3e-4`; final baseline LR: **`5e-4`**.

Transfer models use **differential learning rates**: pretrained backbone `1e-4`, new classifier `5e-4`. The pretrained backbone is updated more conservatively, while the freshly initialized head adapts faster.

`ReduceLROnPlateau` monitors **validation loss**, factor 0.5, patience 2, minimum LR `1e-6`. Validation loss is a smooth plateau signal; the best checkpoint is selected using **validation Macro-F1**, with validation loss as a tie-breaker.

Early-stop patience is **6 for baseline** and **5 for transfer models**. It is implemented as a safeguard, but the selected final runs reached their configured maximum epoch before early stopping terminated training.

In [8]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

dummy_param = nn.Parameter(torch.zeros(1))
optimizer = AdamW([dummy_param], lr=5e-4, betas=(0.9, 0.999), weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6)
print('optimizer: AdamW')
print('betas:', optimizer.defaults['betas'])
print('weight_decay:', optimizer.defaults['weight_decay'])
print('scheduler: ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)')
print('checkpoint metric: validation Macro-F1')

optimizer: AdamW
betas: (0.9, 0.999)
weight_decay: 0.0001
scheduler: ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)
checkpoint metric: validation Macro-F1


## 10. Mini-Batch Training Step

The essential optimization order is `zero_grad → forward → loss → backward → optimizer.step`. PyTorch gradients accumulate by default, therefore gradients are cleared before the next batch. `backward()` computes gradients through backpropagation and `optimizer.step()` updates the parameters. AMP is used on CUDA in the full training scripts.

In [ ]:
def illustrative_training_step(model, images, labels, optimizer, criterion):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

### Sanity check before long training
A 38-image subset was intentionally overfit before long runs. This is an implementation test, not a generalization result. If a network cannot fit a tiny set, the labels, loss, backpropagation or optimizer may be wrong.

In [9]:
print('38-image sanity check')
print('step 30 -> accuracy 71.05%')
print('step 40 -> accuracy 100.00%')
print('Result: PASS')

38-image sanity check
step 30 -> accuracy 71.05%
step 40 -> accuracy 100.00%
Result: PASS


## 11. Transfer Learning Development

ResNet18 and EfficientNet-B0 start from ImageNet-pretrained weights and are **fully fine-tuned** rather than used only as frozen feature extractors. The original classifier is replaced by a dropout layer and a 38-class linear head.

In [10]:
# Exact model builders are defined in src/transfer_models.py and src/efficientnet_models.py.
# Recorded trainable parameter counts from the completed project runs:
print('Custom CNN parameters : 399,142')
print('ResNet18 parameters   : 11,196,006')
print('EfficientNet-B0 params: 4,056,226')
print('Transfer dropout      : 0.30')
print('Backbone LR / head LR : 1e-4 / 5e-4')

Custom CNN parameters : 399,142
ResNet18 parameters   : 11,196,006
EfficientNet-B0 params: 4,056,226
Transfer dropout      : 0.30
Backbone LR / head LR : 1e-4 / 5e-4


### Final training configuration

| Setting | Custom CNN | ResNet18 | EfficientNet-B0 |
|---|---:|---:|---:|
| Batch size | 64 | 32 | 32 |
| Maximum epochs | 15 | 12 | 12 |
| Main/backbone LR | `5e-4` | `1e-4` | `1e-4` |
| Classifier LR | — | `5e-4` | `5e-4` |
| Dropout | 0.40 | 0.30 | 0.30 |
| Weight decay | `1e-4` | `1e-4` | `1e-4` |
| Early-stop patience | 6 | 5 | 5 |

The transfer-learning gain is therefore evaluated against a transparent scratch baseline, while the optimizer family, weight decay and evaluation logic remain controlled.

## 12. Evaluation Metrics

For one class in a one-vs-rest view:

- `Precision = TP / (TP + FP)`
- `Recall / Sensitivity = TP / (TP + FN)`
- `Specificity = TN / (TN + FP)`
- `F1 = 2 × Precision × Recall / (Precision + Recall)`

**Macro-F1** gives all 38 classes equal weight and is therefore used for checkpoint selection. **Weighted-F1** weights each class according to support. ROC-AUC is calculated using a one-vs-rest multiclass formulation. Accuracy alone is not sufficient because it does not reveal class-specific errors.

## 13. Training Dynamics — Project-Specific Curves

The following plots are the project's equivalents of the example presentation's **Loss vs Validation Loss**, **Accuracy vs Validation Accuracy**, and validation-score plots. They use the recorded EfficientNet-B0 model-development/full-training history.

> **Provenance:** these curves explain training methodology and optimization behavior. They are not relabeled as ultra-strict training curves. Final audited ultra-strict test metrics are reported separately.

### Loss vs Validation Loss
![Loss vs validation loss](../outputs/figures/presentation_metrics/01_efficientnet_loss_vs_validation_loss.svg)

### Accuracy vs Validation Accuracy
![Accuracy vs validation accuracy](../outputs/figures/presentation_metrics/02_efficientnet_accuracy_vs_validation_accuracy.svg)

### Validation Macro-F1
![Validation Macro-F1](../outputs/figures/presentation_metrics/03_efficientnet_validation_macro_f1.svg)

In [11]:
history_path = ROOT / 'outputs' / 'histories' / 'efficientnet_b0_b32_blr1e4_hlr5e4_full_history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    best_row = history.loc[history['validation_macro_f1'].idxmax()]
    print('Recorded development history:', len(history), 'epochs')
    print('LR epochs 1-9: 1e-4')
    print('LR epochs 10-12: 5e-5')
    print(f"Best recorded validation Macro-F1: {best_row['validation_macro_f1']:.6f} at epoch {int(best_row['epoch'])}")
else:
    print('History CSV not found in this checkout.')

Recorded development history: 12 epochs
LR epochs 1-9: 1e-4
LR epochs 10-12: 5e-5
Best recorded validation Macro-F1: 0.992400 at epoch 10


The LR reduction visible from epoch 10 illustrates the scheduler methodology. The training curve is not interpreted by one point alone; train and validation behavior are considered together.

## 14. Overfitting / Underfitting Analysis

The deterministic clean-train subset helps separate random augmentation effects from same-domain generalization.

| Model | Clean-train accuracy | Validation accuracy | Final test accuracy |
|---|---:|---:|---:|
| Custom CNN | 77.89% | 83.55% | 84.62% |
| ResNet18 | 98.16% | 98.81% | 97.66% |
| EfficientNet-B0 | 99.74% | 98.68% | 99.01% |

The transfer models do not show the large same-domain train-to-validation collapse expected from severe conventional training-image memorization. EfficientNet has roughly a one-percentage-point clean-train/validation difference and the test is slightly above validation. The Custom CNN is substantially weaker and less stable.

The clean-train set is a balanced 760-image diagnostic subset, so its value should not be overinterpreted as a full-training-set metric.

## 15. Final Ultra-Strict Results

These are the headline results after the stricter split/integrity protocol.

In [12]:
final_results = pd.DataFrame([
    ['Custom CNN', 84.620413, 0.781323, 0.836106, 1647],
    ['ResNet18', 97.656177, 0.968603, 0.976314, 251],
    ['EfficientNet-B0', 99.010178, 0.987373, 0.990114, 106],
    ['50/50 Ensemble', 99.140910, 0.989733, 0.991396, 92],
], columns=['model','accuracy_pct','macro_f1','weighted_f1','errors'])
print('Model                 Accuracy   Macro-F1   Weighted-F1   Errors')
for _, r in final_results.iterrows():
    print(f"{r['model']:<22}{r['accuracy_pct']:7.2f}%     {r['macro_f1']:.4f}       {r['weighted_f1']:.4f}      {int(r['errors']):4d}")

Model                 Accuracy   Macro-F1   Weighted-F1   Errors
Custom CNN             84.62%     0.7813       0.8361      1647
ResNet18               97.66%     0.9686       0.9763       251
EfficientNet-B0        99.01%     0.9874       0.9901       106
50/50 Ensemble         99.14%     0.9897       0.9914        92


Transfer learning produces the largest development gain: **84.62% scratch → 97.66% ResNet18 → 99.01% EfficientNet-B0**. EfficientNet-B0 is the strongest individual model despite using far fewer parameters than ResNet18. The ensemble adds a smaller final improvement to **99.14%**.

## 16. Class-Wise F1, Sensitivity and Specificity

This is the project-specific equivalent of the example project's class-wise F1 / sensitivity / specificity graph. Unlike the training curves above, these values are derived from the **final ultra-strict EfficientNet-B0 locked-test confusion matrix**. Specificity is calculated one-vs-rest as `TN / (TN + FP)`.

![Class-wise F1 sensitivity specificity](../outputs/figures/presentation_metrics/04_efficientnet_classwise_f1_sensitivity_specificity.svg)

In [13]:
cm_path = ROOT / 'outputs' / 'audit' / 'full_control' / 'efficientnet_confusion_matrix.csv'
per_class_path = ROOT / 'outputs' / 'audit' / 'full_control' / 'efficientnet_per_class.csv'

cm_df = pd.read_csv(cm_path, index_col=0)
cm = cm_df.to_numpy(dtype=int)
per_class = pd.read_csv(per_class_path).sort_values('class_index').reset_index(drop=True)

N = int(cm.sum())
tp = np.diag(cm).astype(float)
support = cm.sum(axis=1).astype(float)
pred_support = cm.sum(axis=0).astype(float)
fn = support - tp
fp = pred_support - tp
tn = N - tp - fn - fp
precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)
specificity = np.divide(tn, tn + fp, out=np.zeros_like(tp), where=(tn + fp) != 0)

metrics = pd.DataFrame({
    'class_name': per_class['class_name'],
    'precision': precision,
    'sensitivity_recall': recall,
    'specificity': specificity,
    'f1_score': f1,
    'support': support.astype(int),
})

print(f'Final test samples: {N:,}')
print(f'Correct: {int(tp.sum()):,}')
print(f'Errors: {N-int(tp.sum()):,}')
print(f'Accuracy: {100*tp.sum()/N:.6f}%')
print(f'Macro-F1: {f1.mean():.6f}')
worst = metrics.loc[metrics['f1_score'].idxmin()]
print(f"Lowest class-wise F1: {worst['class_name']} = {worst['f1_score']:.6f}")

Final test samples: 10,709
Correct: 10,603
Errors: 106
Accuracy: 99.010178%
Macro-F1: 0.987373
Lowest class-wise F1: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot = 0.944444


Several classes are classified perfectly on the locked test. The lower-F1 classes are more informative for error analysis. For example, Corn Cercospora/Gray leaf spot, Tomato Late blight, Tomato Early blight, Tomato mosaic virus and Tomato Septoria leaf spot are among the more difficult classes.

## 17. Confusion Matrix and Classification Report

The confusion matrix shows **which classes are confused**, not only how many predictions are wrong. The final EfficientNet matrix is 38×38. The generation script `src/generate_presentation_metrics.py` creates raw and row-normalized matrices, a class-ID legend, two detailed classification-report tables and an overall summary.

In [14]:
pairs = []
names = cm_df.index.tolist()
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i != j and cm[i, j] > 0:
            pairs.append((int(cm[i, j]), names[i], names[j]))

print('Largest off-diagonal confusions')
for count, true_name, pred_name in sorted(pairs, reverse=True)[:5]:
    print(f'{count:2d} : {true_name} -> {pred_name}')

Largest off-diagonal confusions
17 : Tomato___Septoria_leaf_spot -> Tomato___Late_blight
11 : Tomato___Early_blight -> Tomato___Late_blight
 7 : Potato___Late_blight -> Tomato___Late_blight
 7 : Corn_(maize)___Northern_Leaf_Blight -> Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
 5 : Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot -> Corn_(maize)___Northern_Leaf_Blight


In [ ]:
# Regenerate all presentation-aligned evaluation figures from repository outputs.
# Run from the repository root:
# .\.venv\Scripts\python.exe src\generate_presentation_metrics.py

# The resulting folder contains:
# 01 loss vs validation loss
# 02 accuracy vs validation accuracy
# 03 validation Macro-F1
# 04 class-wise F1 / sensitivity / specificity
# 05 raw confusion matrix
# 06 normalized confusion matrix
# 07 class-ID legend
# 08-09 detailed classification report tables
# 10 overall classification summary
# 11 lowest class-wise F1 scores

## 18. Validation-Only Ensemble Selection

The ensemble combines ResNet18 and EfficientNet probabilities using **soft voting**. Candidate weights were selected on validation data only. The test set was not used to select the weight.

In [15]:
ensemble_candidates = [(100,0,0.986237),(75,25,0.988722),(50,50,0.992033),(25,75,0.986400),(0,100,0.982431)]
print('ResNet/EfficientNet   Validation Macro-F1')
for r, e, score in ensemble_candidates:
    marker = '  <- selected' if (r, e) == (50, 50) else ''
    print(f'{r}/{e:<18} {score:.6f}{marker}')

ResNet/EfficientNet   Validation Macro-F1
100/0                 0.986237
75/25                 0.988722
50/50                 0.992033  <- selected
25/75                 0.986400
0/100                 0.982431


The selected probability rule is `Pensemble = 0.5 × PResNet18 + 0.5 × PEfficientNet`. The ensemble improves the final locked-test error count from 106 for EfficientNet to **92**, but the improvement is much smaller than the earlier scratch-to-transfer-learning gain.

## 19. Leakage Audit and Protocol Revision

The historical image-level ensemble reached approximately **99.76%**. Instead of accepting this unusually high score directly, a leakage audit was performed. The original split contained:

- **10** exact cross-split duplicate groups/pairs,
- **68** perceptual near-duplicate pairs under the original audit,
- **4,956** mapped same-physical-leaf cross-split groups.

A stricter protocol was then built. The final strict review found **39** `dHash≤4` cross-split pairs. Using priority `test > validation > train`, the lower-priority counterpart was quarantined: **34 train + 4 validation + 0 test**. The locked test remained unchanged.

Under the implemented protocol there is no detected exact, mapped-leaf or strict `dHash≤4` overlap. This is not a mathematical guarantee that every unmapped physical specimen is unique because mapped-leaf coverage is approximately 75.7%.

The historical-to-final performance difference must not be attributed entirely to leakage because the split protocol and training/evaluation conditions also changed.

In [16]:
print('Original audit: exact=10, near-duplicate=68, mapped same-leaf=4956')
print('Final strict dHash<=4 pairs before quarantine: 39')
print('Quarantine: train=34, validation=4, test=0')
print('Detected strict overlaps remaining: 0')

Original audit: exact=10, near-duplicate=68, mapped same-leaf=4956
Final strict dHash<=4 pairs before quarantine: 39
Quarantine: train=34, validation=4, test=0
Detected strict overlaps remaining: 0


## 20. Reproducibility, Random-Label Control and Uncertainty

Three EfficientNet seeds test whether near-99% same-domain performance is a lucky run. Seed 777 is **not** selected as the headline just because it has the highest test score; doing so after observing test outcomes would be cherry-picking.

In [17]:
seed_runs = [(42,99.010,0.987373,106),(123,98.767,0.983668,132),(777,99.225,0.990149,83)]
for seed, acc, f1_value, errors in seed_runs:
    print(f'seed {seed:<3}: accuracy={acc:.3f}%  Macro-F1={f1_value:.6f}  errors={errors}')
print('mean accuracy = 99.001%')
print('std = 0.229 percentage points')

seed 42  : accuracy=99.010%  Macro-F1=0.987373  errors=106
seed 123 : accuracy=98.767%  Macro-F1=0.983668  errors=132
seed 777 : accuracy=99.225%  Macro-F1=0.990149  errors=83
mean accuracy = 99.001%
std = 0.229 percentage points


### Random-label sanity check
Training labels were shuffled for a negative-control experiment. The chance level for 38 uniform classes is approximately `1/38 = 2.63%`. True-label validation accuracy after shuffled-label training was **1.84%**, Macro-F1 **0.0183**. This supports the absence of trivial label/pipeline leakage, but does not prove that every possible leakage mechanism is impossible.

### Bootstrap uncertainty
A 1,000-sample ordinary bootstrap gives an EfficientNet accuracy 95% interval of approximately **98.81%–99.20%**. This is an ordinary sample bootstrap, not a stratified bootstrap.

## 21. Calibration, Robustness and Grad-CAM

EfficientNet calibration metrics on the locked test: **ECE 0.003845**, **NLL 0.035704**, **Brier 0.016145**. ECE is bin-dependent, so NLL and Brier are also reported.

Representative robustness results:

| Condition | EfficientNet accuracy |
|---|---:|
| Clean | 99.01% |
| Brightness 0.60 | 98.83% |
| Brightness 1.40 | 98.07% |
| Contrast 0.60 | 98.49% |
| JPEG quality 30 | 98.13% |
| Rotation 15° | 99.41% |
| Gaussian blur radius 2 | **84.08%** |
| Large center occlusion | **55.38%** |

Brightness/contrast/JPEG/rotation are relatively mild stresses; blur and large occlusion are substantially more damaging. This supports sensitivity to fine visual details but does not causally prove a shortcut.

Grad-CAM is used as **supportive qualitative evidence only**. It does not prove that the network exclusively uses disease lesions.

Existing full-control figures:

![EfficientNet reliability](../outputs/figures/full_control/efficientnet_reliability.png)

![EfficientNet robustness](../outputs/figures/full_control/efficientnet_robustness_stress.png)

## 22. External PlantDoc Evaluation

PlantDoc is used as a **zero-shot external / out-of-domain stress test**. The mapped subset contains 236 images across 27 mapped source classes. Manual semantic mapping is required, so this is not a directly comparable 38-class benchmark. No PlantDoc retraining/fine-tuning is used for this result.

In [18]:
print('PlantDoc mapped OOD subset: 236 images, 27 mapped source classes')
print('EfficientNet-B0: accuracy=23.31%, mapped Macro-F1=0.2183')
print('50/50 ensemble : accuracy=25.00%, mapped Macro-F1=0.2349')

PlantDoc mapped OOD subset: 236 images, 27 mapped source classes
EfficientNet-B0: accuracy=23.31%, mapped Macro-F1=0.2183
50/50 ensemble : accuracy=25.00%, mapped Macro-F1=0.2349


The large PlantVillage → PlantDoc drop is best described as **strong domain dependence / poor out-of-domain transfer**, not direct proof that individual training images were simply memorized. Within PlantVillage, clean-train, validation and test are close; PlantDoc changes background, framing, illumination, scale and other acquisition characteristics. The correct claim is therefore strong controlled-domain performance with limited external validity.

## 23. Complete Experimental Development Summary

| What changed / was tested | Why | What happened |
|---|---|---|
| Scratch Custom CNN | establish a transparent baseline | 84.62% final accuracy |
| ResNet18 transfer learning | test pretrained feature reuse | 97.66% |
| EfficientNet-B0 | improve parameter efficiency and performance | 99.01%, best individual |
| Dropout 0.20/0.40/0.60 | study regularization strength | 0.60 suppressed learning |
| LR pilots | avoid arbitrary baseline LR | final baseline `5e-4` |
| Differential transfer LR | protect pretrained backbone while adapting head | retained in final transfer setup |
| ReduceLROnPlateau | reduce step size after val-loss plateau | LR reduction visible in recorded development history |
| Validation Macro-F1 checkpoint | balanced 38-class selection | avoids selection by accuracy alone |
| 50/50 ensemble | combine model probabilities | 99.14%, 92 errors |
| Leakage audit | investigate historical 99.76% | exact/near/same-leaf overlap found |
| Ultra-strict protocol | make evaluation more defensible | ensemble still 99.14% |
| Multiple seeds | test lucky-run hypothesis | 99.001% ± 0.229 pp |
| Random-label control | test trivial pipeline leakage | chance-level validation; PASS |
| Calibration/bootstrap | assess confidence and uncertainty | ECE 0.003845; accuracy CI reported |
| PlantDoc | test external validity | 23–25%, strong domain shift |

This table is the core presentation logic: **method → reason → observed effect/evidence**.

## 24. Scientific Claim Discipline

For a defensible presentation/report, the following distinctions are maintained:

- Do not say the test set was 'never seen'; test **images** participated in a model-independent integrity audit. Test labels/predictions were not used for training/tuning/selection.
- Do not claim universal 'zero leakage'; claim zero **detected** exact, mapped-leaf and strict dHash≤4 overlap under the implemented audit.
- Do not say all of the historical 99.76% was caused by leakage.
- Do not use PlantDoc failure as proof of simple training-image memorization; it demonstrates strong domain shift.
- Do not treat Grad-CAM or occlusion as causal proof.
- Do not report seed 777 as the final model simply because it has the highest observed test score.
- Do not interpret near-99% PlantVillage performance as near-99% real-world field performance.

## 25. Reproduction Commands

Environment setup:

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

Final pipelines:

```bat
call .\run_ultrastrict_all.bat
call .\run_full_control_all.bat
```

Presentation-aligned figures:

```powershell
.\.venv\Scripts\python.exe src\generate_presentation_metrics.py
```

## 26. Final Conclusion

The strongest result of the project is not a new architecture; it is the **controlled development process**. A scratch CNN establishes the baseline, regularization and validation-driven optimization define the training methodology, transfer learning produces the largest performance gain, validation-only ensembling gives a smaller improvement, and leakage/OOD audits define the limits of the result.

Under the final audited PlantVillage protocol, EfficientNet-B0 achieves **99.01% accuracy / 0.9874 Macro-F1** and the 50/50 ensemble reaches **99.14% / 0.9897**. Three-seed testing shows that the same-domain result is reproducible. However, PlantDoc accuracy of **23–25%** demonstrates strong domain dependence.

**Final scientific statement:** near-99% PlantVillage accuracy is reproducible under the audited protocol, but it must not be interpreted as near-99% real-world field accuracy.